In [28]:
import json
import os 

with open('config.json', 'r') as f:
    data = json.load(f)

pathway_gen = os.path.abspath(data["python_files"])
pathway_temp = os.path.abspath(data["publications"])
pathway = os.path.join(pathway_temp, "kopp2021small")
original_data_pathway = os.path.join(pathway, "original_data")

complete_path_1 = os.path.join(original_data_pathway, "KoppEbel_2021_AnimBehavCogn_SMES.txt")

out_pathway = os.path.join(pathway, "standardized_data")
if not os.path.exists(out_pathway):
    os.makedirs(out_pathway)

In [29]:
import pandas as pd
import numpy as np
import pyreadstat

df = pd.read_table(complete_path_1)


In [30]:
df['study_id']="kopp2021small"
df.columns = map(str.lower, df.columns)
df=df.applymap(lambda s: s.lower() if type(s) == str else s)
df.rename(columns={"sex": "sex_original",
    "group":"group_id",
    "individual":"ape",
    'obs_day':"day",
    'obs_month':"month",
    'obs_year':"year"}, inplace=True)

In [31]:
comp_path_name_errors = os.path.join(pathway_gen, "common_name_errors.csv")

df_name  = pd.read_csv(comp_path_name_errors)
df['ape'] = df['ape'].str.rstrip()
for x,y in zip(df_name['wrong'],df_name['right']):
    df['ape'].replace(x, y, inplace=True)

comp_path_ape_info = os.path.join(pathway_gen, "apes_includeindatabase.csv")
apedf = pd.read_csv(comp_path_ape_info)    
df= df.merge(apedf,left_on='ape', right_on='name', how='left')


In [32]:
spe_2=[]  #group id based on species list
for index, row in df.iterrows():
    if not pd.isna(row['sex']):
        spe_2.append(row['sex'])
    else:
        spe_2.append(row['sex_original'])
df = df.assign(sex=spe_2)

# df.columns
df.rename(columns={"ape": "participant"}, inplace=True)

sex_original_update = [['female','f'],['male','m']]
for x,y in sex_original_update:
    df['sex'].replace(x, y, inplace=True, regex=True)

In [33]:
spe_2=[]  #group id based on species list
for index, row in df.iterrows():
    if not pd.isna(row['species']):
        spe_2.append(row['species'])
    else:
        spe_2.append('chimpanzee')
df = df.assign(species=spe_2)

In [34]:
decimal_reduce = ['propallmrb_sess', 'propse_sess']
for x in decimal_reduce:
       df[x] = df[x].astype(float).round(2)

group_replace = ['berlin','halle','leipzig','heidelberg','wuppertal', 'magdeburg']
for x in group_replace:
    df['group_id'].replace(x, '', inplace=True, regex=True)

In [35]:
df.rename(columns={"group_id": "species_subgroup",
                   "age_month":"age_original"}, inplace=True)

comp_path_birth_dates = os.path.join(pathway_gen, "apes_age_calculations.csv")
ape_dob = pd.read_csv(comp_path_birth_dates) 

df= df.merge(ape_dob,left_on='participant', right_on='name', how='left') #insert dob of participants
df['dodc'] = df['year'].astype(str) + '-' + df['month'].astype(str) + '-' + df['day'].astype(str)
df['dodc'] = pd.to_datetime(df['dodc'])##convert date of data collection to datetime format
df['dob'] = pd.to_datetime(df['dob'])##convert date of birth to datetime format

df['age_in_years'] = (df['dodc'] - df['dob']).dt.days//365

In [36]:

kopp2021small_standardized=df[['study_id','year', 'month','day', 'participant','age_original', 'age_in_years','sex', 'species',
        'session', 'condition', 'species_subgroup', 'duration_session_ms',
       'duration_condition_ms', 'mrbcount', 'durationallmrb_ms',
       'propallmrb_sess', 'secount', 'durationse_ms', 'propse_sess'
       ]]
comp_out_path_stand = os.path.join(out_pathway, 'kopp2021small_standardized.csv')
kopp2021small_standardized.to_csv(comp_out_path_stand, encoding='utf-8-sig', index=False)


names =kopp2021small_standardized.columns.tolist()
df = pd.DataFrame(names)
df = df.rename(columns={0: "column_name"})
df["description"] = ""
kopp2021small_glossary=df[["column_name", "description"]]

comp_out_path_glossary = os.path.join(out_pathway, 'kopp2021small_glossary.csv')
kopp2021small_glossary.to_csv(comp_out_path_glossary, encoding='utf-8-sig', index=False)
